# Notebook 08 -- Ensamble Final para Producción

---


## Objetivo

Ensamblar sistema MoE final: reentrenamiento con todos los datos, clase wrapper MoEForecaster, API de producción. Demostraración de pronóstico 168h y serialización.


## Entrada

features + modelos optimizados + gating + hiperparámetros


## Salida

Modelo MoE serializado (.pkl/.joblib) + clase MoEForecaster + documentación de API


## Instalación

In [ ]:
# Instalación de dependencias
# En Docker (make up / make pipeline) ya vienen instaladas con versiones fijas
# (requirements.txt). En Google Colab, después de clonar el repo y hacer
# %cd notebooks, se instalan las mismas versiones:
import sys
if "google.colab" in sys.modules:
    !pip install -q -r ../requirements.txt

## Imports y configuración

In [ ]:
# ============================================================
# CONFIGURACIÓN — Modificar según entorno
# ============================================================
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"
# Figuras en inglés (versión para publicación); la carpeta no se versiona
os.makedirs(f"{RESULTS_DIR}/english", exist_ok=True)
RANDOM_SEED = 42
N_JOBS      = -1

MODEL_VERSION = "1.0.0"   # Cambiar con cada reentrenamiento mayor

import os, json, pickle, warnings
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

import holidays

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.fontsize': 10, 'axes.grid': True, 'grid.alpha': 0.3,
})

# Crear directorios de producción
PROD_DIR = Path(MODELS_DIR) / 'production'
EXPERTS_DIR = PROD_DIR / 'experts'
for d in [PROD_DIR, EXPERTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Directorios de producción creados en: {PROD_DIR}')

---
## 1. Carga de Datos Completos y Archivos

In [ ]:
# ============================================================
# Cargar datos completos
# ============================================================
def load_artifact(paths, description='artefacto'):
    """Intentar cargar desde múltiples paths posibles."""
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f'  ✓ {description} cargado de: {p}')
            return p
    print(f'  ⚠️  {description} no encontrado')
    return None

# ---- Features ----
print('Cargando datos...')
features_path = load_artifact(
    [f'{DATA_DIR}/processed/features.parquet'],
    'features.parquet'
)
if features_path:
    df = pd.read_parquet(features_path)
    if not isinstance(df.index, pd.DatetimeIndex):
        for col in ['timestamp','datetime','date']:
            if col in df.columns:
                df = df.set_index(col); break
    df = df.sort_index()
else:
    # Sin datos sintéticos de respaldo: si falta un artefacto de un notebook
    # anterior hay que ejecutarlo, no seguir con datos inventados.
    raise FileNotFoundError("No existe data/processed/features.parquet. Ejecuta antes el notebook 02 (make pipeline).")

In [ ]:
df = df[~df.index.year.isin([2020, 2021])]

In [ ]:
df.head()

In [ ]:
# ---- Clusters ----
cluster_path = load_artifact(
    [f'{DATA_DIR}/clustering/cluster_assignments.parquet',
     f'{DATA_DIR}/processed/cluster_assignments.parquet'],
    'cluster_assignments.parquet'
)
if cluster_path:
    clusters = pd.read_parquet(cluster_path)
    if not isinstance(clusters.index, pd.DatetimeIndex):
        for col in ['timestamp','datetime','date']:
            if col in clusters.columns:
                clusters = clusters.set_index(col); break

clusters = clusters.rename(columns={'cluster_day': 'type_day', 'cluster_week': 'type_week'})

# Cada asignación trae su fecha (columna datetime, ver notebook 03). Reconstruirla
# por posición la desplazaba 48 h, porque el clustering descarta la primera semana ISO incompleta.
clusters['timestamp'] = clusters.index
clusters = clusters.reset_index(drop=True)

print(f"\nForma de los clústeres: {clusters.shape}")
print(f"Columnas de los clústeres: {list(clusters.columns)}")

# Asegurar que timestamp sea datetime en ambos
clusters['timestamp'] = pd.to_datetime(clusters['timestamp'])

clusters.index = clusters['timestamp']
clusters.drop(columns=['timestamp'], inplace=True)

In [ ]:
clusters

In [ ]:
df = df.join(clusters[["expert_id", "type_day", "type_week"]], how="inner")

In [ ]:
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(df.columns)

# Lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

In [ ]:
TARGET = 'demand'
EXCLUDE = [TARGET, 'type_day', 'type_week', 'expert_id']
# FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]
FEATURE_COLS = calendar_lags

print(f'\nDataset completo: {df.shape}')
print(f'Rango: {df.index[0].date()} → {df.index[-1].date()}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Expertos únicos: {df["expert_id"].nunique()}')

expert_ids = sorted(df['expert_id'].dropna().unique())
print(f'\nExpertos: {expert_ids}')

In [ ]:
df.head()

In [ ]:
# ============================================================
# Cargar best_params del HPO (NB07)
# ============================================================

best_params_global = {}
hp_path = load_artifact(
    [f'{MODELS_DIR}/hyperparameters/best_hyperparameters.json'],
    'best_hyperparameters.json'
)

if hp_path:
    with open(hp_path) as f:
        best_params_global = json.load(f)
    print(f'HPO params cargados: {len(best_params_global)} expertos')
else:
    print('  Sin HPO params — se usarán defaults robustos')

# Cargar selección de modelos (NB05)
selection_path = load_artifact(
    [f'{RESULTS_DIR}/05_expert_selection_best.csv'],
    'expert_selection.csv'
)
expert_selection = {}
if selection_path:
    df_sel = pd.read_csv(selection_path)
    if 'expert_id' in df_sel.columns and 'model' in df_sel.columns:
        expert_selection = dict(zip(df_sel['expert_id'], df_sel['model']))
    print(f'Selección cargada: {len(expert_selection)} expertos con modelo asignado')

# Cargar gating params (NB07)
gating_path = load_artifact(
    [f'{MODELS_DIR}/hyperparameters/best_gating_params.json'],
    'gating_params.json'
)
if gating_path:
    with open(gating_path) as f:
        gating_params = json.load(f)
else:
    gating_params = {'threshold': 0.75, 'tau': 1.0}
    print(f'  Usando defaults: {gating_params}')

THRESHOLD = gating_params.get('threshold', 0.75)
TAU       = gating_params.get('tau', 1.0)
print(f'\nGating: threshold={THRESHOLD}, tau={TAU}')

---
## Reentrenamiento con Todos los Datos

### Estrategia de selección de modelo por experto

| Tamaño del clúster | Familia permitida | Razón |
|---|---|---|
| n < 50 | Ridge (regularización máxima) | Sesgo alto, varianza mínima |
| 50 ≤ n < 200 | Lineal (Ridge/Lasso/ElasticNet) | Regularización suficiente |
| 200 ≤ n < 1000 | ML (LightGBM/XGB/RF) | Balance sesgo-varianza |
| n ≥ 1000 | Mejor de todas las familias (del NB05) | Datos suficientes para complejidad |

In [ ]:
# ============================================================
# Fábrica de modelos
# ============================================================

def get_model_for_expert(expert_id, n_samples, best_params=None, selected_model=None):
    """
    Retorna el modelo apropiado para un experto según:
    1. Modelo seleccionado en NB05 (si disponible)
    2. Best params de HPO NB07 (si disponible)
    3. Restricción de complejidad por tamaño de clúster
    """
    params = best_params or {}

    # Restricción por tamaño
    if n_samples < 50:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', Ridge(alpha=params.get('alpha', 100.0), random_state=RANDOM_SEED))
        ])
    elif n_samples < 200:
        alpha = params.get('alpha', 10.0)
        l1_ratio = params.get('l1_ratio', 0.5)
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=2000))
        ])
        return lgb.LGBMRegressor(**lgbm_params)
    else:
        # Clúster grande: mejor modelo del NB05
        model_name = selected_model or 'lgbm'

        if 'bayesian_ridge' in model_name.lower():
            return Pipeline([
                ('scaler', StandardScaler()),
                ('model', BayesianRidge())
            ])
        elif 'lasso' in model_name.lower():
            return Pipeline([
                ('scaler', StandardScaler()),
                ('model', Lasso(alpha=params.get('alpha', 1.0), random_state=RANDOM_SEED))
            ])
        elif 'ridge' in model_name.lower():
            return Pipeline([
                ('scaler', StandardScaler()),
                ('model', Ridge(alpha=params.get('alpha', 1.0), random_state=RANDOM_SEED))
            ])
        elif 'xgb' in model_name.lower():
            xgb_params = {
                'n_estimators': params.get('n_estimators', 500),
                'learning_rate': params.get('learning_rate', 0.05),
                'max_depth': params.get('max_depth', 6),
                'subsample': params.get('subsample', 0.8),
                'colsample_bytree': params.get('colsample_bytree', 0.8),
                'reg_alpha': params.get('reg_alpha', 0.1),
                'reg_lambda': params.get('reg_lambda', 1.0),
                'random_state': RANDOM_SEED, 'verbosity': 0, 'n_jobs': N_JOBS,
            }
            return xgb.XGBRegressor(**xgb_params)
        elif 'random_forest' in model_name.lower() or 'rf' in model_name.lower():
            return RandomForestRegressor(
                n_estimators=params.get('n_estimators', 300),
                max_depth=params.get('max_depth', None),
                min_samples_leaf=params.get('min_samples_leaf', 5),
                random_state=RANDOM_SEED, n_jobs=N_JOBS,
            )
        else:
            # Default: LightGBM
            lgbm_params = {
                'n_estimators': params.get('n_estimators', 500),
                'learning_rate': params.get('learning_rate', 0.05),
                'num_leaves': params.get('num_leaves', 63),
                'min_child_samples': params.get('min_child_samples', 10),
                'subsample': params.get('subsample', 0.8),
                'colsample_bytree': params.get('colsample_bytree', 0.8),
                'reg_alpha': params.get('reg_alpha', 0.01),
                'reg_lambda': params.get('reg_lambda', 0.1),
                'random_state': RANDOM_SEED, 'verbose': -1, 'n_jobs': N_JOBS,
            }
            return lgb.LGBMRegressor(**lgbm_params)

print('Fábrica de modelos lista.')
print('Restricciones de complejidad:')
print('  n <  50  → Ridge')
print('  n < 200  → ElasticNet')
print('  n < 1000 → LightGBM conservador')
print('  n ≥ 1000 → Mejor modelo de NB05 con HPO de NB07')


In [ ]:
# ============================================================
# Entrenar K expertos con todos los datos
# ============================================================

print('=== ENTRENANDO K EXPERTOS CON TODOS LOS DATOS ===')
print(f'Dataset completo: {len(df):,} observaciones\n')

production_experts = {}
expert_training_log = []

df_clean = df.dropna(subset=FEATURE_COLS + [TARGET])

for eid in expert_ids:
    mask = df_clean['expert_id'] == eid
    n_samples = mask.sum()

    if n_samples < 10:
        print(f'  ⚠️  Experto {eid}: solo {n_samples} obs — omitido')
        continue

    X_k = df_clean.loc[mask, FEATURE_COLS].values
    y_k = df_clean.loc[mask, TARGET].values

    # Obtener params del HPO
    params_k = best_params_global.get(f'{expert_ids.index(eid)}', {})
    model_name_k = expert_selection.get(eid, None)

    print(model_name_k)
    print(params_k)

    model_k = get_model_for_expert(eid, n_samples, params_k, model_name_k)
    model_k.fit(X_k, y_k)

    production_experts[eid] = model_k

    # Determinar familia
    if hasattr(model_k, 'steps'):
        familia = type(model_k.steps[-1][1]).__name__
    else:
        familia = type(model_k).__name__

    expert_training_log.append({
        'expert_id': eid, 'n_samples': n_samples,
        'model_family': familia,
        'has_hpo_params': len(params_k) > 0,
    })

    print(f'  ✓ {eid:20s}: n={n_samples:5,}, modelo={familia}')

print(f'\nExpertos entrenados: {len(production_experts)}/{len(expert_ids)}')

df_log = pd.DataFrame(expert_training_log)
display(df_log)

In [ ]:
production_experts

In [ ]:
# ============================================================
# Reentrenar clasificadores con todos los datos
# ============================================================

print('=== ENTRENANDO CLASIFICADORES DE CLÚSTERES ===')

# Features de clasificación (solo calendario — disponibles en tiempo real)
CLF_FEATURES = FEATURE_COLS
print(f'Features para clasificadores: {len(CLF_FEATURES)}')

df_clf = df.dropna(subset=CLF_FEATURES + ['type_day', 'type_week'])
X_clf = df_clf[CLF_FEATURES].values

# Clasificador type_day
from sklearn.preprocessing import LabelEncoder
le_day = LabelEncoder()
y_day = le_day.fit_transform(df_clf['type_day'].astype(str))

# Cargar modelo seleccionado en NB04 si existe
clf_day_path = load_artifact(
    [f'{MODELS_DIR}/classifiers/best_classifier_day.pkl',
     f'{MODELS_DIR}/production/classifier_day.pkl'],
    'best_classifier_day'
)

if clf_day_path:
    with open(clf_day_path, 'rb') as f:
        clf_day_loaded = pickle.load(f)
    # Si es un dict con metadatos (formato NB04)
    if isinstance(clf_day_loaded, dict):
        clf_day_obj = clf_day_loaded.get('model', clf_day_loaded.get('pipeline'))
    else:
        clf_day_obj = clf_day_loaded
    if clf_day_obj is not None:
        # Reentrenar con todos los datos
        clf_day_obj.fit(X_clf, y_day)
        clf_day_prod = clf_day_obj
        print('  ✓ classifier_day reentrenado (modelo NB04)')
    else:
        clf_day_prod = None
else:
    clf_day_prod = None

if clf_day_prod is None:
    # Fallback: LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    clf_day_prod = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, multi_class='multinomial',
                                    C=1.0, random_state=RANDOM_SEED, n_jobs=N_JOBS))
    ])
    clf_day_prod.fit(X_clf, y_day)
    print('  ✓ classifier_day entrenado (LogisticRegression fallback)')

# Clasificador type_week
le_week = LabelEncoder()
y_week = le_week.fit_transform(df_clf['type_week'].astype(str))

clf_week_path = load_artifact(
    [f'{MODELS_DIR}/classifiers/best_classifier_week.pkl',
     f'{MODELS_DIR}/production/classifier_week.pkl'],
    'best_classifier_week'
)

if clf_week_path:
    with open(clf_week_path, 'rb') as f:
        clf_week_loaded = pickle.load(f)
    if isinstance(clf_week_loaded, dict):
        clf_week_obj = clf_week_loaded.get('model', clf_week_loaded.get('pipeline'))
    else:
        clf_week_obj = clf_week_loaded
    if clf_week_obj is not None:
        clf_week_obj.fit(X_clf, y_week)
        clf_week_prod = clf_week_obj
        print('  ✓ classifier_week reentrenado (modelo NB04)')
    else:
        clf_week_prod = None
else:
    clf_week_prod = None

if clf_week_prod is None:
    clf_week_prod = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=1.0,
                                    random_state=RANDOM_SEED, n_jobs=N_JOBS))
    ])
    clf_week_prod.fit(X_clf, y_week)
    print('  ✓ classifier_week entrenado (LogisticRegression fallback)')

print('\nClasificadores listos.')

---
## Clase MoEForecaster (wrapper de producción)

Clase sklearn-compatible que encapsula todo el sistema MoE.

In [ ]:
# ============================================================
# BLOQUE 3: Clase MoEForecaster
# ============================================================
# La clase vive en src/moe_forecaster/forecaster.py para que moe_forecaster.pkl
# se pueda cargar fuera de este notebook: si se define aquí, pickle la guarda como
# __main__.MoEForecaster y solo existe dentro de esta sesión de Jupyter.
import os, sys
sys.path.insert(0, os.path.abspath("../src"))
from moe_forecaster import MoEForecaster
print('Clase MoEForecaster importada desde src/moe_forecaster.')

In [ ]:
# ============================================================
# Crear instancia de producción
# ============================================================

moe_forecaster = MoEForecaster(
    experts=production_experts,
    classifier_day=clf_day_prod,
    classifier_week=clf_week_prod,
    label_encoder_day=le_day,
    label_encoder_week=le_week,
    feature_cols=FEATURE_COLS,
    clf_feature_cols=CLF_FEATURES,
    gating_threshold=THRESHOLD,
    tau=TAU,
    min_prob_soft=0.05,
    version=MODEL_VERSION,
)

card = moe_forecaster.get_model_card()
print('=== MODEL CARD ===')
for k, v in card.items():
    if k != 'expert_ids':
        print(f'  {k}: {v}')
print(f'  expert_ids: {card["expert_ids"]}')

---
## Pronóstico 168h

In [ ]:
# ============================================================
# test
# ============================================================

print('=== PRONÓSTICO 168h ===')

# Usar las últimas 168+lags observaciones del dataset
n_seed = min(500, len(df_clean))
df_seed = df_clean.iloc[-n_seed:]

# Predecir punto a punto (sin recursión completa para el sanity test)
X_seed = df_seed[FEATURE_COLS].values
y_seed = df_seed[TARGET].values

preds_sanity, metadata_sanity = moe_forecaster.predict_with_metadata(df_seed)

# Validaciones de rango
preds_valid = preds_sanity
n_out_of_range = np.sum((preds_valid < 3e6) | (preds_valid > 15e6))
preds_min = preds_valid.min()
preds_max = preds_valid.max()
preds_mean = preds_valid.mean()

print(f'\nValidaciones de rango físico:')
print(f'  Mín predicción:  {preds_min:,.0f} kWh  (esperado > 3,000,000 kWh)  {"✓" if preds_min > 3e6 else "✗"}')
print(f'  Máx predicción:  {preds_max:,.0f} kWh  (esperado < 15,000,000 kWh) {"✓" if preds_max < 15e6 else "✗"}')
print(f'  Media predicción:{preds_mean:,.0f} kWh  (esperado ~7-11 millones) {"✓" if 5e6 < preds_mean < 12e6 else "?"}')
print(f'  Fuera de rango:  {n_out_of_range} obs {"✓" if n_out_of_range == 0 else "⚠️"}')

# Gating stats
mode_counts = metadata_sanity['gating_mode'].value_counts()
print(f'\nDistribución de gating:')
for mode, cnt in mode_counts.items():
    print(f'  {mode}: {cnt} ({100*cnt/len(metadata_sanity):.1f}%)')

# Métricas en el seed (in-sample — solo para sanity)
mape_sanity = 100 * np.mean(np.abs((y_seed - preds_sanity) / (y_seed + 1e-9)))
print(f'\nMAPE in-sample (seed): {mape_sanity:.2f}%')
if mape_sanity < 10:
    print('  ✓ El modelo está aprendiendo (MAPE < 10% in-sample)')
else:
    print('  ⚠️  MAPE in-sample alto — verificar datos/modelos')

# Visualización sanity
n_plot = min(168, len(df_seed))  # 1 semana
idx_plot = df_seed.index[-n_plot:]

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Panel 1: Predicción vs Real
ax1 = axes[0]
ax1.plot(idx_plot, y_seed[-n_plot:], 'k-', lw=1.5, label='Real (XM)', alpha=0.9)
ax1.plot(idx_plot, preds_sanity[-n_plot:], 'b-', lw=1.2, label='MoE Pronóstico', alpha=0.85)
ax1.fill_between(idx_plot,
                  preds_sanity[-n_plot:]*0.97,
                  preds_sanity[-n_plot:]*1.03,
                  alpha=0.2, color='blue', label='±3% banda')
ax1.set_ylabel('Demanda (kWh)')
ax1.set_title('Predicción vs Real (última semana)')
ax1.legend(loc='upper right')

# Panel 2: Error porcentual
ax2 = axes[1]
error_pct = 100 * (preds_sanity[-n_plot:] - y_seed[-n_plot:]) / (y_seed[-n_plot:] + 1e-9)
ax2.plot(idx_plot, error_pct, color='tomato', lw=0.8, alpha=0.8)
ax2.axhline(0, color='k', lw=0.8, ls='--')
ax2.axhline(3, color='orange', lw=0.8, ls=':')
ax2.axhline(-3, color='orange', lw=0.8, ls=':')
ax2.set_ylabel('Error (%)')
ax2.set_xlabel('Fecha')
ax2.set_title('Error porcentual (líneas naranjas = ±3%)')

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR) / '08_fig_sanity_test.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n✓ test completado.')

In [ ]:
# ============================================================
# Sanity test
# ============================================================

print('=== 168h FORECAST ===')

# Use the last 168+lags observed points from the dataset
n_seed = min(500, len(df_clean))
df_seed = df_clean.iloc[-n_seed:]

# Predict point by point (without full recursion for the sanity test)
X_seed = df_seed[FEATURE_COLS].values
y_seed = df_seed[TARGET].values

preds_sanity, metadata_sanity = moe_forecaster.predict_with_metadata(df_seed)

# Range checks
preds_valid = preds_sanity
n_out_of_range = np.sum((preds_valid < 3e6) | (preds_valid > 15e6))
preds_min = preds_valid.min()
preds_max = preds_valid.max()
preds_mean = preds_valid.mean()

print(f'\nPhysical range checks:')
print(f'  Min prediction:   {preds_min:,.0f} kWh  (expected > 3,000,000 kWh)  {"✓" if preds_min > 3e6 else "✗"}')
print(f'  Max prediction:   {preds_max:,.0f} kWh  (expected < 15,000,000 kWh) {"✓" if preds_max < 15e6 else "✗"}')
print(f'  Mean prediction:  {preds_mean:,.0f} kWh  (expected ~7-11 million) {"✓" if 5e6 < preds_mean < 12e6 else "?"}')
print(f'  Out of range:     {n_out_of_range} obs {"✓" if n_out_of_range == 0 else "⚠️"}')

# Gating stats
mode_counts = metadata_sanity['gating_mode'].value_counts()
print(f'\nGating distribution:')
for mode, cnt in mode_counts.items():
    print(f'  {mode}: {cnt} ({100*cnt/len(metadata_sanity):.1f}%)')

# Metrics on the seed (in-sample — only for sanity check)
mape_sanity = 100 * np.mean(np.abs((y_seed - preds_sanity) / (y_seed + 1e-9)))
print(f'\nIn-sample MAPE (seed): {mape_sanity:.2f}%')
if mape_sanity < 10:
    print('  ✓ The model is learning (in-sample MAPE < 10%)')
else:
    print('  ⚠️  High in-sample MAPE — check data/models')

# Sanity visualization
n_plot = min(168, len(df_seed))  # 1 week
idx_plot = df_seed.index[-n_plot:]

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Panel 1: Prediction vs Actual
ax1 = axes[0]
ax1.plot(idx_plot, y_seed[-n_plot:], 'k-', lw=1.5, label='Actual (XM)', alpha=0.9)
ax1.plot(idx_plot, preds_sanity[-n_plot:], 'b-', lw=1.2, label='MoE Forecast', alpha=0.85)
ax1.fill_between(
    idx_plot,
    preds_sanity[-n_plot:] * 0.97,
    preds_sanity[-n_plot:] * 1.03,
    alpha=0.2,
    color='blue',
    label='±3% band'
)
ax1.set_ylabel('Demand (kWh)')
ax1.set_title('Prediction vs Actual (Last Week)')
ax1.legend(loc='upper right')

# Panel 2: Percentage error
ax2 = axes[1]
error_pct = 100 * (preds_sanity[-n_plot:] - y_seed[-n_plot:]) / (y_seed[-n_plot:] + 1e-9)
ax2.plot(idx_plot, error_pct, color='tomato', lw=0.8, alpha=0.8)
ax2.axhline(0, color='k', lw=0.8, ls='--')
ax2.axhline(3, color='orange', lw=0.8, ls=':')
ax2.axhline(-3, color='orange', lw=0.8, ls=':')
ax2.set_ylabel('Error (%)')
ax2.set_xlabel('Date')
ax2.set_title('Percentage Error (orange lines = ±3%)')

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR) / "english" / '08_fig_sanity_test.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n✓ test completed.')

In [ ]:
print("Generando predicciones para enero de 2026...")

# 1. Definir el rango de fechas para la predicción (Enero 2026)
start_date_pred = pd.Timestamp('2026-01-01 00:00:00')
end_date_pred = pd.Timestamp('2026-01-31 23:00:00')
future_index = pd.date_range(start=start_date_pred, end=end_date_pred, freq='H')

# Crear un DataFrame para almacenar las características futuras y predicciones
df_pred_features = pd.DataFrame(index=future_index, columns=FEATURE_COLS + [TARGET])

# Obtener los días festivos de Colombia para un rango relevante
# Incluye años anteriores para 'days_from_prev_holiday' y futuros para 'days_to_next_holiday'
co_holidays = holidays.CO(years=range(2020, 2027))

def generate_features_for_timestamp(ts, historical_df, future_df, feature_cols, target_col, co_holidays):
    """Genera las características para un timestamp dado.
    historical_df: DataFrame con datos históricos reales (hasta el final del entrenamiento)
    future_df: DataFrame con las predicciones ya generadas para el futuro (para lags)
    """
    features = {}

    # 1. Características cíclicas de tiempo
    features['hour_sin'] = np.sin(2 * np.pi * ts.hour / 24)
    features['hour_cos'] = np.cos(2 * np.pi * ts.hour / 24)
    features['dayofweek_sin'] = np.sin(2 * np.pi * ts.dayofweek / 7)
    features['dayofweek_cos'] = np.cos(2 * np.pi * ts.dayofweek / 7)
    features['month_sin'] = np.sin(2 * np.pi * ts.month / 12)
    features['month_cos'] = np.cos(2 * np.pi * ts.month / 12)

    # 2. Características de días festivos
    features['is_holiday'] = 1 if ts.date() in co_holidays else 0

    features['is_bridge'] = 0
    if ts.weekday() == 0 and (ts.date() - timedelta(days=3)) in co_holidays:
        features['is_bridge'] = 1
    if ts.weekday() == 4 and (ts.date() + timedelta(days=3)) in co_holidays:
        features['is_bridge'] = 1
    if (ts.date() - timedelta(days=1)) in co_holidays and (ts.date() + timedelta(days=1)) in co_holidays:
        features['is_bridge'] = 1

    features['days_to_next_holiday'] = 0
    features['days_from_prev_holiday'] = 0

    if ts.date() not in co_holidays:
        next_holiday_date = None
        for day_offset in range(1, 366):
            future_date = ts.date() + timedelta(days=day_offset)
            if future_date in co_holidays:
                next_holiday_date = future_date
                break
        if next_holiday_date:
            features['days_to_next_holiday'] = (next_holiday_date - ts.date()).days
        else:
            features['days_to_next_holiday'] = 365

        prev_holiday_date = None
        for day_offset in range(1, 366):
            past_date = ts.date() - timedelta(days=day_offset)
            if past_date in co_holidays:
                prev_holiday_date = past_date
                break
        if prev_holiday_date:
            features['days_from_prev_holiday'] = (ts.date() - prev_holiday_date).days
        else:
            features['days_from_prev_holiday'] = 365

    # 3. Características de rezago (lags)
    # Asegurarse de que los índices son datetime para concatenar y buscar
    if not isinstance(historical_df.index, pd.DatetimeIndex):
        historical_df.index = pd.to_datetime(historical_df.index)
    if not isinstance(future_df.index, pd.DatetimeIndex):
        future_df.index = pd.to_datetime(future_df.index)

    # Get the last known real demand for imputation in the initial gap
    last_known_demand = historical_df[target_col].iloc[-1]

    # Combinar datos históricos y futuros para la búsqueda de lags
    # Se usa historical_df[target_col] para asegurarse de que solo se toman los valores reales de demanda
    # future_df[target_col].dropna() es crucial para que solo se incluyan las predicciones ya hechas
    combined_data = pd.concat([historical_df[target_col], future_df[target_col].dropna()]).sort_index()

    lag_cols_nums = sorted([int(c.split('_')[-1]) for c in feature_cols if c.startswith('demand_lag_')])
    for lag in lag_cols_nums:
        lag_ts = ts - timedelta(hours=lag)
        if lag_ts in combined_data.index:
            features[f'demand_lag_{lag}'] = combined_data.loc[lag_ts]
        else:
            # If the lag is not available (e.g., falls in the gap between historical_df and current ts
            # or before df_clean.index.min()), impute with the last known historical demand.
            features[f'demand_lag_{lag}'] = last_known_demand

    # Solo incluir características que están en FEATURE_COLS
    return {k: v for k, v in features.items() if k in feature_cols}

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

for i, ts in enumerate(future_index):
    # Generar características para el timestamp actual
    current_features_dict = generate_features_for_timestamp(ts, df_clean, df_pred_features, FEATURE_COLS, TARGET, co_holidays)

    # Convertir a DataFrame de una fila para la predicción
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=FEATURE_COLS)

    # Realizar la predicción
    pred = moe_forecaster.predict(X_current)[0]

    # Almacenar la predicción
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Pronosticado {i + 1}/{len(future_index)} horas para Enero 2026. Última predicción: {ts} -> {pred:,.0f} kWh")

print("\nPredicción de Enero 2026 completada.")

# Visualizar la predicción
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Predicción de Demanda (kWh)', color='blue')
plt.title('Pronóstico de Demanda Eléctrica para Enero 2026')
plt.xlabel('Fecha')
plt.ylabel('Demanda (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Mostrar algunas estadísticas de la predicción
preds_jan_2026 = df_pred_features[TARGET].dropna().values
preds_min = preds_jan_2026.min()
preds_max = preds_jan_2026.max()
preds_mean = preds_jan_2026.mean()

print(f'\nEstadísticas de la predicción para Enero 2026:')
print(f'  Mínima predicción: {preds_min:,.0f} kWh')
print(f'  Máxima predicción: {preds_max:,.0f} kWh')
print(f'  Media predicción:  {preds_mean:,.0f} kWh')
print(f'  Rango físico válido (3,000-15,000 kWh): {"✓" if 3000 < preds_min and preds_max < 15000 else "✗"}')


---
## Serialización Completa

In [ ]:
# ============================================================
# Guardar todos los archivos de producción
# ============================================================

print('=== SERIALIZANDO SISTEMA MoE COMPLETO ===')

# 1. K expertos individuales
print('\n[1/6] Expertos individuales...')
for eid, model in production_experts.items():
    path = EXPERTS_DIR / f'expert_{eid}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f'  ✓ {len(production_experts)} expertos guardados en {EXPERTS_DIR}')

# 2. Clasificadores
print('\n[2/6] Clasificadores...')
clf_pkg_day = {
    'model': clf_day_prod,
    'label_encoder': le_day,
    'feature_cols': CLF_FEATURES,
    'classes': le_day.classes_.tolist(),
    'n_classes': len(le_day.classes_),
}
with open(PROD_DIR / 'classifier_day.pkl', 'wb') as f:
    pickle.dump(clf_pkg_day, f, protocol=pickle.HIGHEST_PROTOCOL)

clf_pkg_week = {
    'model': clf_week_prod,
    'label_encoder': le_week,
    'feature_cols': CLF_FEATURES,
    'classes': le_week.classes_.tolist(),
    'n_classes': len(le_week.classes_),
}
with open(PROD_DIR / 'classifier_week.pkl', 'wb') as f:
    pickle.dump(clf_pkg_week, f, protocol=pickle.HIGHEST_PROTOCOL)
print('  ✓ classifier_day.pkl y classifier_week.pkl')

# 3. Gating params
print('\n[3/6] Gating params...')
gating_config = {
    'threshold': THRESHOLD,
    'tau': TAU,
    'min_prob_soft': 0.05,
    'description': 'threshold: hard/soft switching. tau: temperatura para distancia-softmax.',
}
with open(PROD_DIR / 'gating_params.json', 'w') as f:
    json.dump(gating_config, f, indent=2)
print(f'  ✓ gating_params.json: {gating_config}')

# 4. Normalization params (media y std por clúster para referencia)
print('\n[4/6] Normalization params...')
norm_params = {}
for eid in expert_ids:
    mask = df_clean['expert_id'] == eid
    if mask.sum() > 0:
        y_k = df_clean.loc[mask, TARGET]
        norm_params[eid] = {
            'mean': float(y_k.mean()),
            'std': float(y_k.std()),
            'p05': float(y_k.quantile(0.05)),
            'p95': float(y_k.quantile(0.95)),
            'n_samples': int(mask.sum()),
        }
with open(PROD_DIR / 'normalization_params.json', 'w') as f:
    json.dump(norm_params, f, indent=2)
print(f'  ✓ normalization_params.json: {len(norm_params)} clústeres')

# 5. Feature config
print('\n[5/6] Feature config...')
lag_cols = [c for c in FEATURE_COLS if c.startswith('demand_lag_')]
rolling_cols = [c for c in FEATURE_COLS if c.startswith('demand_rolling_')]
calendar_cols = [c for c in FEATURE_COLS if any(c.startswith(p) for p in
                ['hour_','dow_','month_','is_holiday','is_bridge','days_to','days_from'])]
fourier_cols = [c for c in FEATURE_COLS if c.startswith('fourier_')]

feature_config = {
    'target': TARGET,
    'all_features': FEATURE_COLS,
    'clf_features': CLF_FEATURES,
    'lag_features': lag_cols,
    'rolling_features': rolling_cols,
    'calendar_features': calendar_cols,
    'fourier_features': fourier_cols,
    'n_features': len(FEATURE_COLS),
    'max_lag': max([int(c.split('_')[-1]) for c in lag_cols]) if lag_cols else 0,
    'forecast_horizon_h': 336,
    'leakage_free': True,
    'rolling_center': False,
}
with open(PROD_DIR / 'feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)
print(f'  ✓ feature_config.json')

# 6. Model card completo
print('\n[6/6] Model card...')

# Cargar métricas de evaluación (NB08)
eval_summary = {}
eval_path = Path(RESULTS_DIR) / '08_evaluation_summary.json'
if eval_path.exists():
    with open(eval_path) as f:
        eval_summary = json.load(f)

model_card = {
    'model_name': 'MoEForecaster-SIN-Colombia',
    'version': MODEL_VERSION,
    'created_at': datetime.now().isoformat(),
    'description': 'Mixture-of-Experts desacoplado para pronóstico de demanda eléctrica horaria del SIN colombiano. Horizonte: 336h (14 días).',
    'architecture': {
        'type': 'MoE desacoplado (cluster-then-predict)',
        'n_experts': len(production_experts),
        'clustering': 'Dual: type_day (perfiles 24D) × type_week (perfiles 168D)',
        'gating': f'Adaptativo — hard si min_conf > {THRESHOLD}, soft si baja confianza',
        'expert_families': list(set(
            type(m.steps[-1][1]).__name__ if hasattr(m, 'steps') else type(m).__name__
            for m in production_experts.values()
        )),
    },
    'data': {
        'source': 'API XM — SINERGOX (POST https://servapibi.xm.com.co/hourly)',
        'target': 'DemandaReal (kWh) — SIN Colombia',
        'training_start': str(df_clean.index[0].date()),
        'training_end': str(df_clean.index[-1].date()),
        'n_obs_total': len(df_clean),
        'resolution': 'horaria',
        'horizon_h': 336,
    },
    'features': feature_config,
    'performance': eval_summary if eval_summary else {
        'note': 'Ejecutar NB08 para obtener métricas de evaluación en test set'
    },
    'constraints': {
        'no_climate_variables': True,
        'no_future_data': True,
        'leakage_free': True,
        'rolling_center': False,
    },
    'references': [
        'Diebold & Mariano (1995). Comparing Predictive Accuracy. JBES.',
        'Hyndman & Koehler (2006). MASE. International Journal of Forecasting.',
        'Bergstra & Bengio (2012). Random Search for Hyper-Parameter Optimization. JMLR.',
        'Hong & Shahidehpour (2015). Short-term electric load forecasting. Wiley.',
    ],
    'artifacts': {
        'experts': f'models/production/experts/expert_*.pkl ({len(production_experts)} files)',
        'classifier_day': 'models/production/classifier_day.pkl',
        'classifier_week': 'models/production/classifier_week.pkl',
        'gating_params': 'models/production/gating_params.json',
        'normalization_params': 'models/production/normalization_params.json',
        'feature_config': 'models/production/feature_config.json',
    },
}
with open(PROD_DIR / 'model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False)
print('  ✓ model_card.json')

# 7. MoEForecaster completo serializado
with open(PROD_DIR / 'moe_forecaster.pkl', 'wb') as f:
    pickle.dump(moe_forecaster, f, protocol=pickle.HIGHEST_PROTOCOL)
print('  ✓ moe_forecaster.pkl (objeto completo)')

---
## Visualización Final del Modelo

In [ ]:
# ============================================================
# Arquitectura visual y resumen
# ============================================================

# ---- Figura: distribución de modelos por experto ----
family_counts = {}
for eid, model in production_experts.items():
    if hasattr(model, 'steps'):
        fam = type(model.steps[-1][1]).__name__
    else:
        fam = type(model).__name__
    family_counts[fam] = family_counts.get(fam, 0) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Pie chart familias
ax1 = axes[0]
colors = plt.cm.Set2(np.linspace(0, 1, len(family_counts)))
wedges, texts, autotexts = ax1.pie(
    family_counts.values(),
    labels=family_counts.keys(),
    autopct='%1.0f%%',
    colors=colors, startangle=90,
)
ax1.set_title(f'Distribución de familias de modelos\n({len(production_experts)} expertos)')

# Panel 2: Bar chart n_samples por experto
ax2 = axes[1]
if expert_training_log:
    df_log_plot = df_log.sort_values('n_samples', ascending=False)
    colors2 = ['#2196F3' if n >= 1000 else '#FF9800' if n >= 200 else '#4CAF50' if n >= 50 else '#F44336'
               for n in df_log_plot['n_samples']]
    ax2.barh(df_log_plot['expert_id'], df_log_plot['n_samples'], color=colors2, alpha=0.85)
    ax2.axvline(x=50, color='red', ls='--', lw=0.8, label='50 (min)')
    ax2.axvline(x=200, color='orange', ls='--', lw=0.8, label='200 (ML umbral)')
    ax2.axvline(x=1000, color='blue', ls='--', lw=0.8, label='1000 (DL umbral)')
    ax2.set_xlabel('N observaciones de entrenamiento')
    ax2.set_title('Tamaño de clúster por experto')
    ax2.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR) / '08_fig_model_architecture.png', dpi=300, bbox_inches='tight')
plt.show()

# ---- Resumen final ----
print('\n' + '='*60)
print('SISTEMA MoE — RESUMEN DE PRODUCCIÓN')
print('='*60)
print(f'Versión:          {MODEL_VERSION}')
print(f'Creado:           {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'N expertos:       {len(production_experts)}')
print(f'Gating threshold: {THRESHOLD}')
print(f'Tau:              {TAU}')
print(f'N features:       {len(FEATURE_COLS)}')
print(f'Horizonte:        336h (14 días)')
print(f'Fuente datos:     API XM — SINERGOX')
print()
print('Familias de modelos:')
for fam, cnt in sorted(family_counts.items(), key=lambda x: -x[1]):
    print(f'  {fam:30s}: {cnt} expertos')
print()
print('Archivos generados:')
for name, path in model_card['artifacts'].items():
    print(f'  ✓ {path}')

if eval_summary:
    print(f'\nRendimiento en test set:')
    print(f'  MAPE global:        {eval_summary.get("moe_mape_global", "N/A")}%')
    print(f'  MASE global:        {eval_summary.get("moe_mase_global", "N/A")}')
    print(f'  Mejora vs global:   {eval_summary.get("improvement_vs_global_pct", "N/A")}%')
    target_ok = eval_summary.get('moe_mape_global', 99) < 3.0
    print(f'  Target MAPE < 3%:   {"✅ LOGRADO" if target_ok else "⚠️  PENDIENTE"}')
else:
    print('\n⚠️  Ejecutar NB08 para obtener métricas de test set')

print('='*60)

In [ ]:
# ============================================================
# Architecture visualization and summary
# ============================================================

# ---- Figure: model family distribution per expert ----
family_counts = {}
for eid, model in production_experts.items():
    if hasattr(model, 'steps'):
        fam = type(model.steps[-1][1]).__name__
    else:
        fam = type(model).__name__
    family_counts[fam] = family_counts.get(fam, 0) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Pie chart (model families)
ax1 = axes[0]
colors = plt.cm.Set2(np.linspace(0, 1, len(family_counts)))
wedges, texts, autotexts = ax1.pie(
    family_counts.values(),
    labels=family_counts.keys(),
    autopct='%1.0f%%',
    colors=colors,
    startangle=90,
)
ax1.set_title(f'Model Family Distribution\n({len(production_experts)} experts)')

# Panel 2: Bar chart (training samples per expert)
ax2 = axes[1]
if expert_training_log:
    df_log_plot = df_log.sort_values('n_samples', ascending=False)
    colors2 = [
        '#2196F3' if n >= 1000 else
        '#FF9800' if n >= 200 else
        '#4CAF50' if n >= 50 else
        '#F44336'
        for n in df_log_plot['n_samples']
    ]

    ax2.barh(df_log_plot['expert_id'], df_log_plot['n_samples'], color=colors2, alpha=0.85)
    ax2.axvline(x=50, color='red', ls='--', lw=0.8, label='50 (min)')
    ax2.axvline(x=200, color='orange', ls='--', lw=0.8, label='200 (ML threshold)')
    ax2.axvline(x=1000, color='blue', ls='--', lw=0.8, label='1000 (DL threshold)')
    ax2.set_xlabel('Number of training samples')
    ax2.set_title('Cluster size per expert')
    ax2.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR) / "english" / '08_fig_model_architecture.png', dpi=300, bbox_inches='tight')
plt.show()

# ---- Final summary ----
print('\n' + '='*60)
print('MoE SYSTEM — PRODUCTION SUMMARY')
print('='*60)
print(f'Version:           {MODEL_VERSION}')
print(f'Created:           {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Number of experts: {len(production_experts)}')
print(f'Gating threshold:  {THRESHOLD}')
print(f'Tau:               {TAU}')
print(f'Number of features:{len(FEATURE_COLS)}')
print(f'Horizon:           336h (14 days)')
print(f'Data source:       XM API — SINERGOX')
print()

print('Model families:')
for fam, cnt in sorted(family_counts.items(), key=lambda x: -x[1]):
    print(f'  {fam:30s}: {cnt} experts')

print()
print('Generated artifacts:')
for name, path in model_card['artifacts'].items():
    print(f'  ✓ {path}')

if eval_summary:
    print(f'\nTest set performance:')
    print(f'  Global MAPE:       {eval_summary.get("moe_mape_global", "N/A")}%')
    print(f'  Global MASE:       {eval_summary.get("moe_mase_global", "N/A")}')
    print(f'  Improvement vs global: {eval_summary.get("improvement_vs_global_pct", "N/A")}%')
    target_ok = eval_summary.get('moe_mape_global', 99) < 3.0
    print(f'  Target MAPE < 3%:  {"✅ ACHIEVED" if target_ok else "⚠️  PENDING"}')
else:
    print('\n⚠️  Run NB08 to obtain test set metrics')

print('='*60)

In [ ]:
# ============================================================
# VERIFICACIÓN FINAL: cargar y usar el modelo serializado
# ============================================================

print('=== VERIFICACIÓN: cargando moe_forecaster.pkl ===')

with open(PROD_DIR / 'moe_forecaster.pkl', 'rb') as f:
    moe_loaded = pickle.load(f)

# Test de predicción
n_test_rows = min(24, len(df_clean))
X_verify = df_clean.iloc[-n_test_rows:]
preds_verify = moe_loaded.predict(X_verify)

print(f'\nPredicciones de verificación (últimas {n_test_rows}h):')
print(f'  Media: {preds_verify.mean():,.0f} kWh')
print(f'  Mín:   {preds_verify.min():,.0f} kWh')
print(f'  Máx:   {preds_verify.max():,.0f} kWh')
print(f'  Rango físico válido: {"✓" if 3000 < preds_verify.min() and preds_verify.max() < 15000 else "✗"}')
print(f'\n✓ El modelo serializado funciona correctamente.')
print('\n🚀 Sistema MoE listo para despliegue en producción.')